Dependências
- Instala o kagglehub, caso necessário
- Carrega bibliotecas necessárias para execução

In [ ]:
!pip install kagglehub[pandas-datasets]

In [ ]:
# Dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter
from typing import Tuple

# Pre-Processamento
import os
from pathlib import Path
from PIL import Image
from keras.preprocessing import image
import numpy as np
from keras.applications.imagenet_utils import preprocess_input
import random

# Train Test
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
import numpy as np
from sklearn.preprocessing import LabelEncoder

Carregar Dataset
- Primeiramente, baixa o dataset na maquina pelo kagglehub

In [ ]:
file_path = "HAM10000_metadata.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "kmader/skin-cancer-mnist-ham10000",
  file_path,
)

root = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print(df['dx'].unique()) # Coluna de previsão
print(df.head())

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
['bkl' 'nv' 'df' 'mel' 'vasc' 'bcc' 'akiec']
     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


Pré-Processsamento das Imagens
- Aplica transformações em cada uma das imagens, principalmente para vetorização
- Carrega os caminhos de todos os arquivos (separados em pastas part1 e part2)
- Ao carregar os caminhos, logo já randomiza o vetor de caminhos para garantir randomização no treinamento e teste posterior
- Faz o vinculo imagem vetorizada x respectiva classe de cancer
- Como a RAM do Google Colab é limitada, faz-se necessário a aplicação executar em batches, salvo na pasta batches/x_batch_i.npy (e respectivo y_batch_y.npy na mesma pasta)

In [ ]:
# Carrega a imagem, retorna ela vetorizada
def get_image(
      path: str,
      color_mode: str='rgb',
      target_size: Tuple[int, int]=(224, 224)
    ) -> Tuple[Image.Image, np.ndarray]:
  img = image.load_img(path=path, color_mode=color_mode, target_size=target_size)
  x = image.img_to_array(img)
  x = np.expand_dims(x, axis=0)
  x = preprocess_input(x)
  return img, x

In [ ]:
# Carrega todas as imagens, e randomiza a ordem delas (para não repetir o treinamento)
images_path = [os.path.join(dp, f) for dp, dn, filenames in os.walk(root) for f in filenames if os.path.splitext(f)[1].lower() in ['.jpg','.png','.jpeg']]
random.shuffle(images_path) # Garantia de geração randomica para as batches
images_path[0:5] # Procurar em df['image_id'] para achar a classe

['/kaggle/input/skin-cancer-mnist-ham10000/HAM10000_images_part_2/ISIC_0032586.jpg',
 '/kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0025435.jpg',
 '/kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0029033.jpg',
 '/kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0028114.jpg',
 '/kaggle/input/skin-cancer-mnist-ham10000/ham10000_images_part_1/ISIC_0028697.jpg']

In [ ]:
# Criação de vinculo Imagem x Categoria em batches
batch_size = 2000

os.makedirs(f"batches", exist_ok=True)
image_lookup = dict(zip(df['image_id'], df['dx'])) # Facilita a busca da relação Id da imagem x Classe

# Batch por Batch, vai salvando X_batch (imagem vetorizada) e Y_batch (classe da imagem)
for i in range(0, len(images_path), batch_size):
  X_batch, Y_batch = [], []
  image_path_batched = images_path[i:i+batch_size]

  for image_path in image_path_batched:
    img, x = get_image(image_path)
    image_id = Path(image_path).stem # Limpa o caminho, sobra só ISIC_N

    if image_id in image_lookup:
      label = image_lookup[image_id]
      X_batch.append(x[0])
      Y_batch.append(label)

  np.save(f"batches/x_batch_{i//batch_size}.npy", np.array(X_batch))
  np.save(f"batches/y_batch_{i//batch_size}.npy", np.array(Y_batch))
  print(f"Batch {i//batch_size} salva.")
  del X_batch, Y_batch

Batch 0 salva.
Batch 1 salva.
Batch 2 salva.
Batch 3 salva.
Batch 4 salva.
Batch 5 salva.
Batch 6 salva.
Batch 7 salva.
Batch 8 salva.
Batch 9 salva.
Batch 10 salva.


Treinamento da Rede
- Apos salvar em batches, carrega-se as 8 primeiras batches para treino (lembrando que os arquivos das batches já estão randomizados)
- Faz o mapeamento das classes para numeros (ex: 'bkl':0, 'nv':1...)
- Treina o modelo em si (nesse exemplo, o modelo VGG16 pré-treinado com a imagenet)
- Depois, utiliza-se as 2 ultimas batches para teste, apresentando a acurária e perda

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = Flatten()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(7, activation='softmax')(x)
model = Model(inputs, x)

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

le = LabelEncoder()
all_labels = []
for i in range(8):
    y = np.load(f"batches/y_batch_{i}.npy")
    all_labels.extend(y)
le.fit(all_labels)

print("Classes mapeadas:", dict(zip(le.classes_, range(len(le.classes_)))))

Classes mapeadas: {np.str_('akiec'): 0, np.str_('bcc'): 1, np.str_('bkl'): 2, np.str_('df'): 3, np.str_('mel'): 4, np.str_('nv'): 5, np.str_('vasc'): 6}


In [ ]:
# 5 Epocs
for epoch in range(5):
    print(f"Epoch {epoch+1}/10")
    total_loss, total_acc = 0, 0

    # 1 epoch treina com os 8 batches
    for i in range(8):
        print(f"  Batch {i+1}/8")
        X = np.load(f"batches/x_batch_{i}.npy")
        X = preprocess_input(X)
        y = le.transform(np.load(f"batches/y_batch_{i}.npy"))

        hist = model.fit(X, y, epochs=1, batch_size=32, verbose=0)
        total_loss += hist.history['loss'][0]
        total_acc += hist.history['accuracy'][0]

        del X, y

    # Relatório da época
    print(f"Epoch {epoch+1}: Loss={total_loss/8:.4f}, Acc={total_acc/8:.4f}")

Epoch 1/10
  Batch 1/8
  Batch 2/8
  Batch 3/8
  Batch 4/8
  Batch 5/8


In [ ]:
X_test = np.concatenate([
    np.load("batches/x_batch_09.npy") / 255.0,
    np.load("batches/x_batch_10.npy") / 255.0
])
y_test = np.concatenate([
    np.load("batches/y_batch_09.npy"),
    np.load("batches/y_batch_10.npy")
])

loss, acc = model.evaluate(X_test, y_test)
print(f"Acuracia = {acc:.4f}, Loss = {loss:.4f}")